# SAM2 Robot Arm Inference — Interactive Point Prompting

**Deployment notebook**: no GT masks needed — click points on frame 0 to prompt the model.

### Usage
1. Set paths in **Cell 1** and run Cells 1–3.
2. Run **Cell 4** — an interactive figure appears.
   - Toggle **ARM / GRIPPER** button to select the active object.
   - **Left-click** on the image to add a positive point (include in mask).
   - **Right-click** to add a negative point (exclude from mask).
   - The mask preview updates after every click.
   - Click **Reset** to start over.
3. When satisfied, run **Cell 5** to propagate through the entire video.
4. Run **Cell 6** to visualise key frames.
5. *(Optional)* Run **Cell 7** to save masks to disk.

> **Requires**: `pip install ipympl`  
> Restart the kernel after installing if it was just added.

In [ ]:
# Cell 1 — Imports & Paths
# ipympl enables interactive %matplotlib widget. Install once: pip install ipympl
%matplotlib widget
import os
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import torch
import ipywidgets as widgets
from IPython.display import display

# ── Adjust to your environment ─────────────────────────────────────────────
SAM2_ROOT = "/path/to/sam2_repo"   # e.g. "/home/haoxiang/sam2"
sys.path.insert(0, SAM2_ROOT)

CKPT_PATH = "/data/haoxiang/logs/sam2_finetune/airexo_task0013/checkpoints/checkpoint.pt"

# Directory containing per-frame jpg files named 00000.jpg, 00001.jpg, ...
VIDEO_DIR = "/data/haoxiang/data/airexo2_processed/sam2_finetune/JPEGImages/scene_0001"
# ──────────────────────────────────────────────────────────────────────────

print(f"Video dir : {VIDEO_DIR}")
print(f"Checkpoint: {CKPT_PATH}")
print(f"Checkpoint exists: {os.path.exists(CKPT_PATH)}")
frame_paths = sorted(glob.glob(os.path.join(VIDEO_DIR, "*.jpg")))
print(f"Frames found: {len(frame_paths)}")

In [ ]:
# Cell 2 — Load fine-tuned model & initialise inference state
# NOTE: use the model-architecture config (sam2.1/), NOT the training config (sam2.1_training/)
from sam2.build_sam import build_sam2_video_predictor

predictor = build_sam2_video_predictor(
    config_file="configs/sam2.1/sam2.1_hiera_b+.yaml",
    ckpt_path=CKPT_PATH,
    device="cuda",
    mode="eval",
)

inference_state = predictor.init_state(video_path=VIDEO_DIR)
print("Model loaded, inference state initialised.")

In [ ]:
# Cell 3 — Helper functions

ARM_COLOR = (0.0, 0.6, 1.0)   # blue
GRP_COLOR = (1.0, 0.3, 0.0)   # orange
OBJ_COLORS = {1: ARM_COLOR, 2: GRP_COLOR}


def load_frame(video_dir, frame_idx):
    """Load RGB frame as np.ndarray (H, W, 3) uint8."""
    path = os.path.join(video_dir, f"{frame_idx:05d}.jpg")
    return np.array(Image.open(path).convert("RGB"))


def overlay_mask(image, mask, color, alpha=0.45):
    """Overlay a boolean mask on an RGB image with a given RGB color tuple (0-1 range)."""
    out = image.copy().astype(float)
    for c, v in enumerate(color[:3]):
        out[..., c] = np.where(mask, out[..., c] * (1 - alpha) + v * 255 * alpha, out[..., c])
    return out.clip(0, 255).astype(np.uint8)


print("Helper functions defined.")

In [ ]:
# Cell 4 — Interactive annotation on frame 0
#
# Controls:
#   [ARM / GRIPPER] toggle  — select which object to annotate
#   Left-click              — positive point (foreground)
#   Right-click             — negative point (background)
#   [Reset] button          — clear all points and start over
#
# The mask preview refreshes automatically after each click.

frame0 = load_frame(VIDEO_DIR, 0)
H, W = frame0.shape[:2]

# ── Shared state ──────────────────────────────────────────────────────────
click_state = {
    'points': {1: [], 2: []},   # obj_id -> list of [x, y]
    'labels': {1: [], 2: []},   # obj_id -> list of int (1=pos, 0=neg)
}
cached_masks = {1: None, 2: None}   # last predicted mask per object

# ── Widgets ───────────────────────────────────────────────────────────────
obj_toggle = widgets.ToggleButtons(
    options=[("ARM  (obj=1)", 1), ("GRIPPER  (obj=2)", 2)],
    value=1,
    description="Active object:",
    style={"button_width": "140px", "description_width": "initial"},
)
reset_btn = widgets.Button(
    description="Reset",
    button_style="danger",
    layout=widgets.Layout(width="100px"),
)
status_label = widgets.Label(value="Click on the image to add points.")

# ── Figure ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
plt.subplots_adjust(top=0.95)
ax.axis("off")


def _get_mask_for_obj(oid):
    """Run SAM2 on frame 0 for all objects that have points and return masks."""
    predictor.reset_state(inference_state)
    for o in [1, 2]:
        if click_state['points'][o]:
            pts = np.array(click_state['points'][o], dtype=np.float32)
            lbs = np.array(click_state['labels'][o], dtype=np.int32)
            with torch.inference_mode():
                predictor.add_new_points_or_box(
                    inference_state,
                    frame_idx=0,
                    obj_id=o,
                    points=pts,
                    labels=lbs,
                )
    # Get predictions for all objects at once by re-querying frame 0
    masks_out = {}
    for o in [1, 2]:
        if click_state['points'][o]:
            pts = np.array(click_state['points'][o], dtype=np.float32)
            lbs = np.array(click_state['labels'][o], dtype=np.int32)
            with torch.inference_mode():
                _, obj_ids, mask_logits = predictor.add_new_points_or_box(
                    inference_state,
                    frame_idx=0,
                    obj_id=o,
                    points=pts,
                    labels=lbs,
                )
            # find the index for this obj_id
            if o in obj_ids:
                idx = list(obj_ids).index(o)
                masks_out[o] = mask_logits[idx].squeeze().cpu().numpy() > 0.0
    return masks_out


def refresh_display():
    """Redraw the figure with current masks and point markers."""
    ax.clear()
    ax.axis("off")

    # Start from raw frame and accumulate overlays
    composite = frame0.copy()
    masks = _get_mask_for_obj(None)   # computes for all objects
    for oid, color in OBJ_COLORS.items():
        if oid in masks:
            cached_masks[oid] = masks[oid]
        if cached_masks[oid] is not None:
            composite = overlay_mask(composite, cached_masks[oid], color)

    ax.imshow(composite)

    # Draw point markers
    for oid, color in OBJ_COLORS.items():
        for (x, y), lbl in zip(click_state['points'][oid], click_state['labels'][oid]):
            marker = "*" if lbl == 1 else "x"
            msize  = 14  if lbl == 1 else 12
            mew    = 1.5 if lbl == 1 else 2.5
            ax.plot(x, y, marker, color=color, markersize=msize, markeredgewidth=mew,
                    markeredgecolor="white")

    # Legend
    legend_handles = [
        mpatches.Patch(color=ARM_COLOR, label="arm (obj=1)"),
        mpatches.Patch(color=GRP_COLOR, label="gripper (obj=2)"),
    ]
    ax.legend(handles=legend_handles, loc="upper right", fontsize=10)
    ax.set_title("Frame 0  |  ★ = positive   ✕ = negative", fontsize=11)
    fig.canvas.draw_idle()


def on_click(event):
    if event.inaxes != ax or event.xdata is None:
        return
    oid = obj_toggle.value
    lbl = 1 if event.button == 1 else 0   # left = pos, right = neg
    click_state['points'][oid].append([event.xdata, event.ydata])
    click_state['labels'][oid].append(lbl)
    n_pos = sum(l == 1 for l in click_state['labels'][oid])
    n_neg = sum(l == 0 for l in click_state['labels'][oid])
    obj_name = "arm" if oid == 1 else "gripper"
    status_label.value = (
        f"{obj_name}: {n_pos} positive, {n_neg} negative points | "
        f"arm total={len(click_state['points'][1])}  "
        f"gripper total={len(click_state['points'][2])}"
    )
    refresh_display()


def on_reset(b):
    click_state['points'] = {1: [], 2: []}
    click_state['labels'] = {1: [], 2: []}
    cached_masks[1] = None
    cached_masks[2] = None
    predictor.reset_state(inference_state)
    ax.clear()
    ax.axis("off")
    ax.imshow(frame0)
    ax.set_title("Frame 0  |  ★ = positive   ✕ = negative", fontsize=11)
    status_label.value = "Points cleared. Click to start again."
    fig.canvas.draw_idle()


fig.canvas.mpl_connect("button_press_event", on_click)
reset_btn.on_click(on_reset)

# Initial render
ax.imshow(frame0)
ax.set_title("Frame 0  |  ★ = positive   ✕ = negative", fontsize=11)

display(
    widgets.VBox([
        obj_toggle,
        widgets.HBox([reset_btn, status_label]),
        fig.canvas,
    ])
)

In [ ]:
# Cell 5 — Propagate through the entire video
# Run this AFTER you are happy with the points in Cell 4.

assert any(click_state['points'][1]) or any(click_state['points'][2]), \
    "No points found. Please click on the image in Cell 4 first."

# Re-initialise and re-add all confirmed points
predictor.reset_state(inference_state)
for oid in [1, 2]:
    if click_state['points'][oid]:
        with torch.inference_mode():
            predictor.add_new_points_or_box(
                inference_state,
                frame_idx=0,
                obj_id=oid,
                points=np.array(click_state['points'][oid], dtype=np.float32),
                labels=np.array(click_state['labels'][oid], dtype=np.int32),
            )

# Propagate forward
all_masks = {}   # frame_idx -> {obj_id: bool mask (H, W)}
with torch.inference_mode():
    for f_idx, obj_ids, mask_logits in predictor.propagate_in_video(inference_state):
        all_masks[f_idx] = {
            oid: (mask_logits[i].squeeze().cpu().numpy() > 0.0)
            for i, oid in enumerate(obj_ids)
        }

print(f"Propagated {len(all_masks)} frames.")
print(f"Objects tracked: {sorted({oid for m in all_masks.values() for oid in m})}")

In [ ]:
# Cell 6 — Visualise key frames
%matplotlib inline
import matplotlib
matplotlib.rcParams["figure.dpi"] = 100

n_frames  = len(all_masks)
vis_idxs  = sorted(all_masks.keys())
step      = max(1, n_frames // 4)
VIS_FRAMES = [vis_idxs[0], vis_idxs[step], vis_idxs[2 * step], vis_idxs[-1]]

n_cols = len(VIS_FRAMES)
fig, axes = plt.subplots(3, n_cols, figsize=(5 * n_cols, 12))
if n_cols == 1:
    axes = axes[:, np.newaxis]

for col, f_idx in enumerate(VIS_FRAMES):
    img      = load_frame(VIDEO_DIR, f_idx)
    arm_pred = all_masks.get(f_idx, {}).get(1, np.zeros(img.shape[:2], bool))
    grp_pred = all_masks.get(f_idx, {}).get(2, np.zeros(img.shape[:2], bool))

    axes[0, col].imshow(img)
    axes[0, col].set_title(f"Frame {f_idx}")
    axes[1, col].imshow(overlay_mask(img, arm_pred, ARM_COLOR))
    axes[1, col].set_title("arm (pred)")
    axes[2, col].imshow(overlay_mask(img, grp_pred, GRP_COLOR))
    axes[2, col].set_title("gripper (pred)")

for ax_ in axes.flatten():
    ax_.axis("off")

legend = [
    mpatches.Patch(color=ARM_COLOR, label="arm"),
    mpatches.Patch(color=GRP_COLOR, label="gripper"),
]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=12)
plt.suptitle(f"{os.path.basename(VIDEO_DIR)} — SAM2 point-prompted", fontsize=14)
plt.tight_layout(rect=[0, 0.04, 1, 0.98])
plt.show()

In [ ]:
# Cell 7 (optional) — Save masks to disk

OUT_DIR = "/tmp/seg_output"   # change as needed
os.makedirs(os.path.join(OUT_DIR, "arm"),     exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "gripper"), exist_ok=True)

for f_idx, masks in sorted(all_masks.items()):
    h_ref, w_ref = frame0.shape[:2]
    arm_mask = (masks.get(1, np.zeros((h_ref, w_ref), bool)) * 255).astype(np.uint8)
    grp_mask = (masks.get(2, np.zeros((h_ref, w_ref), bool)) * 255).astype(np.uint8)
    Image.fromarray(arm_mask).save(os.path.join(OUT_DIR, "arm",     f"{f_idx:05d}.png"))
    Image.fromarray(grp_mask).save(os.path.join(OUT_DIR, "gripper", f"{f_idx:05d}.png"))

print(f"Saved {len(all_masks)} frames to {OUT_DIR}/arm/ and {OUT_DIR}/gripper/")